<a href="https://colab.research.google.com/github/RatanakamonS/Stock_Price/blob/main/Optimization_Portfolio_%5BMV_vs_CVaR%5D%5BAllow_Short%26Long_only%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

from scipy.optimize import minimize
from scipy.optimize import linprog
from scipy.stats import norm

In [2]:
# =========================================================
# 0) Helper: robust price extractor (yfinance รองรับ MultiIndex)
# =========================================================
def get_price_series(yf_df: pd.DataFrame) -> pd.Series:
    if yf_df is None or len(yf_df) == 0:
        raise ValueError("yfinance returned empty data.")

    # MultiIndex columns (บางครั้ง yfinance คืนมาเป็น multiindex)
    if isinstance(yf_df.columns, pd.MultiIndex):
        # พยายามหา level 0 ก่อน
        lv0 = list(map(str, yf_df.columns.get_level_values(0)))
        if "Adj Close" in lv0:
            s = yf_df["Adj Close"]
            return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s
        if "Close" in lv0:
            s = yf_df["Close"]
            return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s

        # ถ้าไม่เจอ ให้หา level 1
        lv1 = list(map(str, yf_df.columns.get_level_values(1)))
        if "Adj Close" in lv1:
            s = yf_df.xs("Adj Close", axis=1, level=1)
            return s.iloc[:, 0]
        if "Close" in lv1:
            s = yf_df.xs("Close", axis=1, level=1)
            return s.iloc[:, 0]

        raise KeyError("Cannot find 'Adj Close' or 'Close' in yfinance MultiIndex columns.")

    # Single index columns
    if "Adj Close" in yf_df.columns:
        return yf_df["Adj Close"]
    if "Close" in yf_df.columns:
        return yf_df["Close"]

    raise KeyError("Cannot find 'Adj Close' or 'Close' in yfinance columns.")


In [3]:
# =========================================================
# 1) Load assets (cleaned adjusted close from GitHub)
# =========================================================
URL_ASSETS = (
    "https://raw.githubusercontent.com/RatanakamonS/Stock_Price/"
    "bf86f765ebc2be3aedb395cfb06bdc6fd37e72b7/"
    "3yrs_clean_sp500_adjusted_close_prices.csv"
)

print("Loading asset prices from GitHub...")
prices = pd.read_csv(URL_ASSETS, index_col=0, parse_dates=True, dayfirst=True)
prices = prices.apply(pd.to_numeric, errors="raise")

asset_ret = prices.pct_change().dropna()
tickers = asset_ret.columns.tolist()

start = asset_ret.index.min()
end   = asset_ret.index.max()

print(f"Assets loaded: N={asset_ret.shape[1]}, T(full)={asset_ret.shape[0]}, range={start.date()} to {end.date()}")


Loading asset prices from GitHub...
Assets loaded: N=495, T(full)=752, range=2022-11-01 to 2025-10-30


In [4]:
# =========================================================
# 2) Load market index (^GSPC) and align dates
# =========================================================
print("Loading market index (^GSPC) from yfinance...")
gspc_df = yf.download("^GSPC", start=start, end=end + pd.Timedelta(days=1), progress=False)
gspc_px = get_price_series(gspc_df).dropna()
gspc_ret = gspc_px.pct_change().dropna()

common_dates = asset_ret.index.intersection(gspc_ret.index)
asset_ret = asset_ret.loc[common_dates]
gspc_ret  = gspc_ret.loc[common_dates]

T, N = asset_ret.shape
mu_market = float(gspc_ret.mean())

print(f"Aligned data: N={N}, T={T}")
print(f"mu_market (^GSPC) = {mu_market:.10f}")


Loading market index (^GSPC) from yfinance...


/tmp/ipython-input-4016036512.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  gspc_df = yf.download("^GSPC", start=start, end=end + pd.Timedelta(days=1), progress=False)


Aligned data: N=495, T=751
mu_market (^GSPC) = 0.0008092511


In [5]:
# =========================================================
# 3) Parameters + moments
# =========================================================
beta = 0.95
tail = 1 - beta

mu = asset_ret.mean().values            # (N,)
Sigma = asset_ret.cov().values          # (N,N)
R = asset_ret.values                    # (T,N)

In [6]:
# =========================================================
# 4) Mean–Variance objective
# =========================================================
def mv_variance(w):
    return float(w @ Sigma @ w)


def solve_mv(long_only: bool):
    """
    Mean–Variance:
      min  w' Σ w
      s.t. sum(w) = 1
           mu'w   = mu_market   (EQUAL ตามอาจารย์)
           (optional) w_i >= 0
    """
    cons = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        {"type": "eq", "fun": lambda w: (mu @ w) - mu_market},
    ]

    w0 = np.ones(N) / N

    if long_only:
        bounds = [(0.0, 1.0)] * N  # ใส่ upper=1.0 เพื่อช่วย solver (ไม่ใส่ก็ได้)
    else:
        bounds = [(None, None)] * N

    res = minimize(
        mv_variance, w0,
        constraints=cons,
        bounds=bounds,
        method="SLSQP",
        options={"maxiter": 20000, "ftol": 1e-12}
    )

    return res


def normal_based_var_cvar_loss(port_ret: np.ndarray, beta: float):
    """
    Normal-based VaR/CVaR of LOSS = -Return
    """
    tail = 1 - beta
    mu_p = float(port_ret.mean())
    sd_p = float(port_ret.std(ddof=1))

    z = norm.ppf(beta)
    VaR_loss  = (-mu_p) + sd_p * z
    CVaR_loss = (-mu_p) + sd_p * (norm.pdf(z) / tail)
    return VaR_loss, CVaR_loss

In [7]:
# =========================================================
# 5) CVaR LP Builder (Rockafellar–Uryasev) with EQUAL return constraint
#    Variables: x = [w(1..N), u(1..T), alpha]
#
#    minimize: alpha + (1/(T*tail)) * sum(u_t)
#    s.t.      u_t >= -r_t'w - alpha
#              u_t >= 0
#              sum(w) = 1
#              mu'w  = mu_market      (EQUAL ตามอาจารย์)
#              (optional) w_i >= 0  (long-only)
# =========================================================
def build_cvar_lp_equal_return(R, mu, mu_market, T, N, tail, long_only=False):
    # objective: alpha + (1/(T*tail))*sum(u)
    c = np.zeros(N + T + 1)
    c[N:N+T] = 1.0 / (T * tail)   # u part
    c[-1] = 1.0                   # alpha

    # A_ub x <= b_ub
    # u_t >= -r_t'w - alpha  <=>  -r_t'w - u_t - alpha <= 0
    A_ub = np.zeros((T, N + T + 1))
    b_ub = np.zeros(T)

    for t in range(T):
        A_ub[t, :N]      = -R[t, :]
        A_ub[t, N + t]   = -1.0     # -u_t
        A_ub[t, -1]      = -1.0     # -alpha
        b_ub[t] = 0.0

    # A_eq x = b_eq
    # (1) sum(w)=1
    # (2) mu'w = mu_market
    A_eq = np.zeros((2, N + T + 1))
    b_eq = np.zeros(2)

    A_eq[0, :N] = 1.0
    b_eq[0] = 1.0

    A_eq[1, :N] = mu
    b_eq[1] = float(mu_market)

    # bounds
    if long_only:
        w_bounds = [(0.0, None)] * N
    else:
        w_bounds = [(None, None)] * N

    u_bounds = [(0.0, None)] * T
    alpha_bounds = [(None, None)]

    bounds = w_bounds + u_bounds + alpha_bounds
    return c, A_ub, b_ub, A_eq, b_eq, bounds


def unpack_lp_solution(x, N, T):
    w = x[:N]
    u = x[N:N+T]
    alpha = x[-1]
    return w, u, alpha


def solver_report(name, success, message, w, mu, mu_market, long_only=False, tol=1e-6):
    print("\n" + "="*80)
    print(f"Solver Report: {name}")
    print(f"success : {success}")
    print(f"message : {message}")

    if (not success) or (w is None):
        print("="*80)
        return

    sum_w = float(np.sum(w))
    ret_w = float(mu @ w)
    min_w = float(np.min(w))

    budget_ok = abs(sum_w - 1.0) <= tol
    return_ok = abs(ret_w - float(mu_market)) <= tol
    long_ok   = (min_w + tol) >= 0.0 if long_only else True

    print("-"*80)
    print(f"sum(w)          = {sum_w:.10f} | budget_ok = {budget_ok}")
    print(f"mu @ w          = {ret_w:.10f}")
    print(f"mu_market       = {float(mu_market):.10f}")
    print(f"abs(diff)       = {abs(ret_w - float(mu_market)):.10e} | return_eq_ok = {return_ok}")
    print(f"min(w)          = {min_w:.10f} | long_only_ok = {long_ok}")
    print(f"FEASIBLE(manual)= {bool(budget_ok and return_ok and long_ok)}")
    print("="*80)

In [8]:
# =========================================================
# 6) MODEL 1: Mean–Variance (Allow Short) with mu'w = mu_market
# =========================================================
res_mv_allow = solve_mv(long_only=False)

print("\n" + "="*80)
print("Solver Report: Mean–Variance (Allow Short)")
print(f"success : {res_mv_allow.success}")
print(f"message : {res_mv_allow.message}")
print("="*80)

if not res_mv_allow.success:
    raise RuntimeError("MV (Allow Short) optimization failed. Try checking feasibility: mu_market may be unreachable.")

w_mv_allow = res_mv_allow.x
var_mv_allow = mv_variance(w_mv_allow)

Rp_mv_allow = R @ w_mv_allow
VaR_mv_allow_loss, CVaR_mv_allow_loss = normal_based_var_cvar_loss(Rp_mv_allow, beta)
print(f"var_mv_allow    = {var_mv_allow:.10f}")


Solver Report: Mean–Variance (Allow Short)
success : True
message : Optimization terminated successfully
var_mv_allow    = 0.0000073308


In [9]:
# =========================================================
# 7) MODEL 4: Mean–Variance (Long-only) with mu'w = mu_market
# =========================================================
res_mv_long = solve_mv(long_only=True)

print("\n" + "="*80)
print("Solver Report: Mean–Variance (Long-only)")
print(f"success : {res_mv_long.success}")
print(f"message : {res_mv_long.message}")
print("="*80)

if not res_mv_long.success:
    raise RuntimeError("MV (Long-only) optimization failed. Try checking feasibility: mu_market may be unreachable under long-only.")

w_mv_long = res_mv_long.x
var_mv_long = mv_variance(w_mv_long)

Rp_mv_long = R @ w_mv_long
VaR_mv_long_loss, CVaR_mv_long_loss = normal_based_var_cvar_loss(Rp_mv_long, beta)



Solver Report: Mean–Variance (Long-only)
success : True
message : Optimization terminated successfully


In [10]:
# =========================================================
# 8) MODEL 2: CVaR (Allow Short) with mu'w = mu_market
# =========================================================
c, A_ub, b_ub, A_eq, b_eq, bounds_allow = build_cvar_lp_equal_return(
    R=R, mu=mu, mu_market=mu_market, T=T, N=N, tail=tail, long_only=False
)

res_cvar_allow = linprog(
    c, A_ub=A_ub, b_ub=b_ub,
    A_eq=A_eq, b_eq=b_eq,
    bounds=bounds_allow, method="highs"
)

w_cvar_allow = u_allow = alpha_allow = None
if res_cvar_allow.success:
    w_cvar_allow, u_allow, alpha_allow = unpack_lp_solution(res_cvar_allow.x, N, T)

solver_report(
    "CVaR (Allow Short)", res_cvar_allow.success, res_cvar_allow.message,
    w_cvar_allow, mu, mu_market, long_only=False
)

VaR_allow_loss  = float(alpha_allow) if res_cvar_allow.success else np.nan
CVaR_allow_loss = float(res_cvar_allow.fun) if res_cvar_allow.success else np.nan




Solver Report: CVaR (Allow Short)
success : True
message : Optimization terminated successfully. (HiGHS Status 7: Optimal)
--------------------------------------------------------------------------------
sum(w)          = 1.0000000000 | budget_ok = True
mu @ w          = 0.0008092511
mu_market       = 0.0008092511
abs(diff)       = 2.7321894747e-17 | return_eq_ok = True
min(w)          = -0.5290306818 | long_only_ok = True
FEASIBLE(manual)= True


In [11]:
# =========================================================
# 9) MODEL 3: CVaR (Long-only) with mu'w = mu_market
# =========================================================
c2, A_ub2, b_ub2, A_eq2, b_eq2, bounds_long = build_cvar_lp_equal_return(
    R=R, mu=mu, mu_market=mu_market, T=T, N=N, tail=tail, long_only=True
)

res_cvar_long = linprog(
    c2, A_ub=A_ub2, b_ub=b_ub2,
    A_eq=A_eq2, b_eq=b_eq2,
    bounds=bounds_long, method="highs"
)

w_cvar_long = u_long = alpha_long = None
if res_cvar_long.success:
    w_cvar_long, u_long, alpha_long = unpack_lp_solution(res_cvar_long.x, N, T)

solver_report(
    "CVaR (Long-only)", res_cvar_long.success, res_cvar_long.message,
    w_cvar_long, mu, mu_market, long_only=True
)

VaR_long_loss  = float(alpha_long) if res_cvar_long.success else np.nan
CVaR_long_loss = float(res_cvar_long.fun) if res_cvar_long.success else np.nan



Solver Report: CVaR (Long-only)
success : True
message : Optimization terminated successfully. (HiGHS Status 7: Optimal)
--------------------------------------------------------------------------------
sum(w)          = 1.0000000000 | budget_ok = True
mu @ w          = 0.0008092511
mu_market       = 0.0008092511
abs(diff)       = 1.8431436932e-18 | return_eq_ok = True
min(w)          = 0.0000000000 | long_only_ok = True
FEASIBLE(manual)= True


In [15]:
# =========================================================
# 10) Summary tables
# =========================================================
summary = pd.DataFrame({
    "Model": [
        "Mean–Variance (Allow Short)",
        "CVaR (Allow Short)",
        "CVaR (Long-only)",
        "Mean–Variance (Long-only)"
    ],
    "Objective": [
        var_mv_allow,          # variance
        CVaR_allow_loss,       # CVaR loss
        CVaR_long_loss,        # CVaR loss
        var_mv_long            # variance
    ],
    "VaR_0.95_Loss": [
        VaR_mv_allow_loss,     # normal-based (MV allow)
        VaR_allow_loss,        # alpha from LP
        VaR_long_loss,         # alpha from LP
        VaR_mv_long_loss       # normal-based (MV long)
    ],
    "CVaR_0.95_Loss": [
        CVaR_mv_allow_loss,    # normal-based
        CVaR_allow_loss,       # LP objective
        CVaR_long_loss,        # LP objective
        CVaR_mv_long_loss      # normal-based
    ],
    "Solver_success": [
        bool(res_mv_allow.success),
        bool(res_cvar_allow.success),
        bool(res_cvar_long.success),
        bool(res_mv_long.success),
    ],
    "Solver_message": [
        str(res_mv_allow.message),
        str(res_cvar_allow.message),
        str(res_cvar_long.message),
        str(res_mv_long.message),
    ]
})

weights = pd.DataFrame({
    "Ticker": tickers,
    "w_MV_allow": w_mv_allow,
    "w_CVaR_allow": (w_cvar_allow if res_cvar_allow.success else np.nan),
    "w_CVaR_long":  (w_cvar_long  if res_cvar_long.success  else np.nan),
    "w_MV_long": w_mv_long,
})

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(summary.to_string(index=False))

print("\n" + "="*80)
print("WEIGHTS")
print("="*80)
print(weights.to_string(index=False))

# optional save
summary.to_csv("summary_objectives.csv", index=False)
weights.to_csv("weights_all_models.csv", index=False)


SUMMARY
                      Model  Objective  VaR_0.95_Loss  CVaR_0.95_Loss  Solver_success                                                  Solver_message
Mean–Variance (Allow Short)   0.000007       0.003644        0.004776            True                            Optimization terminated successfully
         CVaR (Allow Short)   0.001513       0.001513        0.001513            True Optimization terminated successfully. (HiGHS Status 7: Optimal)
           CVaR (Long-only)   0.009866       0.007646        0.009866            True Optimization terminated successfully. (HiGHS Status 7: Optimal)
  Mean–Variance (Long-only)   0.000029       0.008122        0.010391            True                            Optimization terminated successfully

WEIGHTS
Ticker  w_MV_allow  w_CVaR_allow  w_CVaR_long    w_MV_long
   MMM   -0.019904      0.006406     0.000000 8.379692e-20
   AOS   -0.027776      0.017904     0.000000 9.399508e-20
   ABT   -0.024077      0.006783     0.000000 2.785935e